# LeetCode 629 — K Inverse Pairs Array bằng Python

- Link: https://leetcode.com/problems/k-inverse-pairs-array/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP), Tổng tiền tố (prefix sum), Cửa sổ trượt trên DP, Tổ hợp, Modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def kInversePairs(self, n: int, k: int) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bài này là ví dụ kinh điển của **tối ưu DP bằng tổng tiền tố**: công thức chuyển trạng thái tự nhiên là "tổng của một đoạn liên tiếp các ô hàng trước", nên mỗi ô có thể tính trong `O(1)` thay vì `O(n)`.

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **cặp nghịch thế (inverse pair / inversion)** và cách đếm.
2. Tìm ra công thức DP bằng ý tưởng **"chèn số lớn nhất vào hoán vị nhỏ hơn"**.
3. Tối ưu từ `O(n · k · n)` xuống `O(n · k)` bằng **prefix sum**, rồi bằng **công thức trượt** `dp[i][j] = dp[i][j-1] + dp[i-1][j] - dp[i-1][j-i]`.
4. Rút bộ nhớ còn `O(k)` và xử lý **modulo khi có phép trừ** đúng cách.
5. Kiểm chứng bằng brute force (liệt kê hoán vị) trên mọi `n` nhỏ.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Với mảng số nguyên `nums`, một **cặp nghịch thế** là cặp chỉ số `(i, j)` với `0 <= i < j < len(nums)` và `nums[i] > nums[j]` (số lớn đứng trước số nhỏ).

Cho hai số `n` và `k`, đếm số **hoán vị** của các số `1, 2, ..., n` có **đúng `k`** cặp nghịch thế. Kết quả lớn nên trả về modulo `10^9 + 7`.

**Ví dụ 1**

```text
Input:  n = 3, k = 0
Output: 1
Giải thích: chỉ có [1,2,3] (tăng dần) là không có cặp nghịch thế nào.
```

**Ví dụ 2**

```text
Input:  n = 3, k = 1
Output: 2
Giải thích: [1,3,2] (cặp (3,2)) và [2,1,3] (cặp (2,1)).
```

**Ràng buộc**

- `1 <= n <= 1000`
- `0 <= k <= 1000`

In [1]:
from itertools import permutations

def count_inversions(arr):
    return sum(1 for i in range(len(arr)) for j in range(i + 1, len(arr)) if arr[i] > arr[j])

for p in permutations([1, 2, 3]):
    print(p, "->", count_inversions(p), "cặp nghịch thế")

(1, 2, 3) -> 0 cặp nghịch thế
(1, 3, 2) -> 1 cặp nghịch thế
(2, 1, 3) -> 1 cặp nghịch thế
(2, 3, 1) -> 2 cặp nghịch thế
(3, 1, 2) -> 2 cặp nghịch thế
(3, 2, 1) -> 3 cặp nghịch thế


## 3. Kiến thức nền

### 3.1. Đếm cặp nghịch thế bằng hai vòng lặp

Với mỗi `i`, đếm số `j > i` mà `arr[j] < arr[i]`. Độ phức tạp `O(n^2)` — đủ dùng cho brute force.

In [2]:
arr = [3, 1, 4, 2]
pairs = [(arr[i], arr[j]) for i in range(4) for j in range(i + 1, 4) if arr[i] > arr[j]]
print(pairs, len(pairs))

[(3, 1), (3, 2), (4, 2)] 3


### 3.2. Số cặp nghịch thế lớn nhất

Hoán vị **giảm dần** `[n, n-1, ..., 1]` có **mọi** cặp là nghịch thế: `n(n-1)/2` cặp. Vậy nếu `k > n(n-1)/2` thì đáp án là 0.

In [3]:
for n in range(1, 7):
    print(n, "->", n * (n - 1) // 2, "| kiểm tra:", count_inversions(list(range(n, 0, -1))))

1 -> 0 | kiểm tra: 0
2 -> 1 | kiểm tra: 1
3 -> 3 | kiểm tra: 3
4 -> 6 | kiểm tra: 6
5 -> 10 | kiểm tra: 10
6 -> 15 | kiểm tra: 15


### 3.3. Tổng tiền tố (prefix sum)

`pre[j] = a[0] + ... + a[j-1]`. Tổng đoạn `a[l..r]` = `pre[r+1] - pre[l]` trong `O(1)`.

In [4]:
a = [3, 1, 4, 1, 5, 9, 2]
pre = [0]
for x in a:
    pre.append(pre[-1] + x)
l, r = 2, 5
print(pre, "| tổng a[2..5] =", pre[r + 1] - pre[l], "=", sum(a[l:r + 1]))

[0, 3, 4, 8, 9, 14, 23, 25] | tổng a[2..5] = 19 = 19


### 3.4. Modulo với phép trừ

`(a - b) % MOD` trong Python **luôn không âm** (khác C++/Java). Tuy vậy, khi `a`, `b` đã được lấy modulo, `a - b` có thể âm — Python vẫn xử lý đúng nhờ `%`, nhưng đừng quên lấy `%` **sau** phép trừ.

In [5]:
MOD = 10**9 + 7
a, b = 5, MOD - 3                  # a "thật" có thể lớn hơn b, nhưng sau khi mod thì a < b
print((a - b) % MOD, (a - b))      # Python: kết quả % luôn trong [0, MOD)

8 -999999999


### 3.5. Chèn phần tử vào list và đếm số nghịch thế mới

`lst.insert(pos, x)` chèn `x` vào vị trí `pos`. Nếu `x` **lớn hơn mọi phần tử** đã có, thì số nghịch thế mới tạo ra = số phần tử đứng **sau** `x` = `len(lst) - pos`.

In [6]:
base = [2, 1, 3]                     # 1 nghịch thế
for pos in range(len(base) + 1):
    lst = base[:]
    lst.insert(pos, 4)
    print(f"chèn 4 vào vị trí {pos}: {lst} -> {count_inversions(lst)} = 1 + {len(base) - pos}")

chèn 4 vào vị trí 0: [4, 2, 1, 3] -> 4 = 1 + 3
chèn 4 vào vị trí 1: [2, 4, 1, 3] -> 3 = 1 + 2
chèn 4 vào vị trí 2: [2, 1, 4, 3] -> 2 = 1 + 1
chèn 4 vào vị trí 3: [2, 1, 3, 4] -> 1 = 1 + 0


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Xây hoán vị bằng cách chèn số lớn nhất

Mọi hoán vị của `1..i` đều tạo ra **duy nhất** từ một hoán vị của `1..i-1` bằng cách chèn số `i` vào một trong `i` vị trí. Vì `i` lớn hơn tất cả, chèn vào vị trí cách cuối `t` ô sẽ thêm **đúng `t`** nghịch thế (`t = 0..i-1`), và **không thay đổi** các nghịch thế cũ.

### 4.2. Công thức DP

Gọi `dp[i][j]` = số hoán vị của `1..i` có đúng `j` nghịch thế.

```text
dp[1][0] = 1  (và dp[1][j] = 0 với j > 0)
dp[i][j] = Σ_{t=0}^{min(j, i-1)} dp[i-1][j-t]
```

### 4.3. Tính tay bảng nhỏ

| i \ j | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| 1 | 1 | 0 | 0 | 0 | 0 | 0 | 0 |
| 2 | 1 | 1 | 0 | 0 | 0 | 0 | 0 |
| 3 | 1 | 2 | 2 | 1 | 0 | 0 | 0 |
| 4 | 1 | 3 | 5 | 6 | 5 | 3 | 1 |

Ví dụ `dp[4][3] = dp[3][3] + dp[3][2] + dp[3][1] + dp[3][0] = 1 + 2 + 2 + 1 = 6` (cửa sổ dài `i = 4`).
`dp[4][5] = dp[3][5] + dp[3][4] + dp[3][3] + dp[3][2] = 0 + 0 + 1 + 2 = 3`.

Mỗi hàng là một **cửa sổ trượt dài `i`** trượt trên hàng trước — đây chính là gợi ý cho tối ưu. (Bảng này còn có tên là **số Mahonian**; mỗi hàng đối xứng và tổng hàng `i` bằng `i!`.)

In [7]:
from collections import Counter
for n in range(1, 5):
    cnt = Counter(count_inversions(p) for p in permutations(range(1, n + 1)))
    print(n, [cnt[j] for j in range(7)], "| tổng =", sum(cnt.values()))

1 [1, 0, 0, 0, 0, 0, 0] | tổng = 1
2 [1, 1, 0, 0, 0, 0, 0] | tổng = 2
3 [1, 2, 2, 1, 0, 0, 0] | tổng = 6
4 [1, 3, 5, 6, 5, 3, 1] | tổng = 24


### 4.4. Nhìn tận mắt phép chèn: từ hoán vị của `1..2` sang `1..3`

Mỗi hoán vị của `[1, 2]` sinh ra 3 hoán vị của `[1, 2, 3]` (chèn số 3 vào 3 vị trí). Bảng dưới cho thấy số nghịch thế **tăng đúng `t`** = số phần tử đứng sau số 3:

| gốc (inv) | chèn ở cuối (t=0) | giữa (t=1) | đầu (t=2) |
|---|---|---|---|
| `[1,2]` (0) | `[1,2,3]` (0) | `[1,3,2]` (1) | `[3,1,2]` (2) |
| `[2,1]` (1) | `[2,1,3]` (1) | `[2,3,1]` (2) | `[3,2,1]` (3) |

Đếm theo cột số nghịch thế: 0 → 1 hoán vị, 1 → 2, 2 → 2, 3 → 1 — đúng hàng `i = 3` của bảng.

In [8]:
def grow(perms_with_inv, i):
    """Từ danh sách (hoán vị của 1..i-1, số nghịch thế) sinh ra danh sách cho 1..i bằng cách chèn i."""
    out = []
    for perm, inv in perms_with_inv:
        L = len(perm)
        for t in range(i):                     # t = số phần tử đứng sau i
            new = perm[:L - t] + [i] + perm[L - t:]
            out.append((new, inv + t))
    return out

level = [([1], 0)]
for i in range(2, 4):
    level = grow(level, i)
for perm, inv in level:
    assert inv == count_inversions(perm)      # số nghịch thế tính theo công thức = đếm thật
    print(perm, inv)
print("phân bố:", Counter(inv for _, inv in level))

[1, 2, 3] 0
[1, 3, 2] 1
[3, 1, 2] 2
[2, 1, 3] 1
[2, 3, 1] 2
[3, 2, 1] 3
phân bố: Counter({1: 2, 2: 2, 0: 1, 3: 1})


## 5. Cách 1 — Brute Force: liệt kê mọi hoán vị

**Ý tưởng.** Sinh `n!` hoán vị, đếm nghịch thế của từng cái, đếm số hoán vị có đúng `k`.

In [9]:
def k_inverse_brute(n: int, k: int) -> int:
    return sum(1 for p in permutations(range(1, n + 1)) if count_inversions(p) == k) % (10**9 + 7)

print(k_inverse_brute(3, 0), k_inverse_brute(3, 1), k_inverse_brute(4, 3))
assert k_inverse_brute(3, 0) == 1
assert k_inverse_brute(3, 1) == 2

1 2 6


**Độ phức tạp:** Time `O(n! · n^2)`, Space `O(n)`. `n = 9` đã ~ 362 nghìn hoán vị × 36 phép so sánh; `n = 1000` là không tưởng.

**Nút thắt:** ta phân biệt từng hoán vị, trong khi chỉ cần biết **có bao nhiêu** hoán vị của `1..i-1` có mỗi số nghịch thế — nhờ tính chất "chèn số lớn nhất" ở mục 4.

In [10]:
import time
for n in (6, 7, 8):
    t = time.perf_counter()
    k_inverse_brute(n, n)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=6: 0.001s
n=7: 0.007s
n=8: 0.065s


## 6. Bước đột phá

**Insight 1 — Chèn số lớn nhất.** Như mục 4: `dp[i][j] = Σ_{t=0}^{min(j,i-1)} dp[i-1][j-t]`. Đây là bài toán con lặp lại → DP `O(n · k · n)`.

**Insight 2 — Tổng đoạn liên tiếp.** Vế phải là tổng **đoạn** `dp[i-1][j-i+1 .. j]` (cắt ở 0). Dùng prefix sum của hàng `i-1` → mỗi ô `O(1)`.

**Insight 3 — Công thức trượt.** So sánh `dp[i][j]` và `dp[i][j-1]`: cửa sổ dịch sang phải một ô — **thêm** `dp[i-1][j]`, **bớt** `dp[i-1][j-i]` (ô vừa rời khỏi cửa sổ, nếu `j - i >= 0`):

```text
dp[i][j] = dp[i][j-1] + dp[i-1][j] - (dp[i-1][j-i] nếu j >= i)
```

**Quy tắc nếu…thì…:**
- Nếu `k > n(n-1)/2` thì đáp án 0 (trả sớm).
- Nếu `j >= i` thì phải trừ ô rời cửa sổ; ngược lại cửa sổ chạm biên 0, không trừ.
- Nếu có phép trừ trong modulo thì lấy `% MOD` sau cùng (Python cho kết quả không âm).

In [11]:
# Kiểm chứng Insight 3 trên bảng nhỏ (tính bằng công thức gốc)
def table_naive(n, K):
    dp = [[0] * (K + 1) for _ in range(n + 1)]
    dp[1][0] = 1
    for i in range(2, n + 1):
        for j in range(K + 1):
            dp[i][j] = sum(dp[i - 1][j - t] for t in range(min(j, i - 1) + 1))
    return dp

T = table_naive(6, 12)
ok = all(T[i][j] == (T[i][j - 1] if j else 0) + T[i - 1][j] - (T[i - 1][j - i] if j >= i else 0)
         for i in range(2, 7) for j in range(13))
print("công thức trượt đúng trên bảng 6 x 12:", ok)

công thức trượt đúng trên bảng 6 x 12: True


## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Brute force (mục 5) | `O(n! · n^2)` | `O(n)` |
| 2 | DP trực tiếp (tổng `i` ô) | `O(n · k · min(n, k))` | `O(k)` |
| 3 | DP + prefix sum | `O(n · k)` | `O(k)` |
| 4 | DP công thức trượt, 1 mảng | `O(n · k)` | `O(k)` |

### 7.1. Cách 2 — DP trực tiếp

**Giải thích.** Cài đúng công thức ở mục 4.2, chỉ giữ hàng trước (`prev`) và hàng hiện tại (`cur`).

In [12]:
MOD = 10**9 + 7

def k_inverse_naive(n: int, k: int) -> int:
    prev = [1] + [0] * k                      # hàng i = 1
    for i in range(2, n + 1):
        cur = [0] * (k + 1)
        for j in range(k + 1):
            s = 0
            for t in range(min(j, i - 1) + 1):    # chèn i cách cuối t ô
                s += prev[j - t]
            cur[j] = s % MOD
        prev = cur
    return prev[k]

print(k_inverse_naive(3, 0), k_inverse_naive(3, 1), k_inverse_naive(4, 3), k_inverse_naive(10, 20))
assert k_inverse_naive(4, 3) == 6

1 2 6 230131


**Đọc code:**

- `prev = [1] + [0]*k`: hoán vị `[1]` có 0 nghịch thế.
- Vòng `t`: thử mọi vị trí chèn số `i`; `t` không được vượt `j` (không có số nghịch thế âm) và `i - 1` (chỉ có `i` vị trí).
- Cuối cùng `prev` là hàng `n`.

**Code nộp LeetCode (Cách 2):** đúng nhưng `n = k = 1000` cần ~`5 · 10^8` phép cộng → **TLE** trong Python. Chỉ để hiểu công thức.

In [13]:
class SolutionNaive:
    def kInversePairs(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        prev = [1] + [0] * k
        for i in range(2, n + 1):
            cur = [0] * (k + 1)
            for j in range(k + 1):
                cur[j] = sum(prev[j - t] for t in range(min(j, i - 1) + 1)) % MOD
            prev = cur
        return prev[k]

assert SolutionNaive().kInversePairs(3, 1) == 2
print("SolutionNaive OK (chỉ cho n, k nhỏ)")

SolutionNaive OK (chỉ cho n, k nhỏ)


**Độ phức tạp Cách 2:** Time `O(n · k · min(n, k))`, Space `O(k)`.

### 7.2. Cách 3 — DP + prefix sum

**Giải thích.** Tính `pre[j] = prev[0] + ... + prev[j-1]` cho hàng trước. Khi đó

```text
cur[j] = Σ_{x = max(0, j-i+1)}^{j} prev[x] = pre[j+1] - pre[max(0, j-i+1)]
```

In [14]:
def k_inverse_prefix(n: int, k: int) -> int:
    if k > n * (n - 1) // 2:
        return 0
    prev = [1] + [0] * k
    for i in range(2, n + 1):
        pre = [0] * (k + 2)
        for j in range(k + 1):
            pre[j + 1] = (pre[j] + prev[j]) % MOD
        cur = [0] * (k + 1)
        for j in range(k + 1):
            lo = max(0, j - i + 1)
            cur[j] = (pre[j + 1] - pre[lo]) % MOD
        prev = cur
    return prev[k]

print(k_inverse_prefix(3, 0), k_inverse_prefix(3, 1), k_inverse_prefix(4, 3), k_inverse_prefix(10, 20))
assert k_inverse_prefix(10, 20) == k_inverse_naive(10, 20)

1 2 6 230131


**Đọc code:**

- `pre` có `k + 2` phần tử: `pre[0] = 0`, `pre[j+1]` = tổng `prev[0..j]`.
- `lo = max(0, j - i + 1)`: đầu trái cửa sổ dài `i` (bị cắt ở 0).
- `(pre[j+1] - pre[lo]) % MOD`: vì `pre` đã lấy modulo, hiệu có thể âm — `%` của Python đưa về `[0, MOD)`.
- Trả sớm 0 khi `k` vượt số nghịch thế tối đa.

**Code nộp LeetCode (Cách 3):**

In [15]:
from itertools import accumulate

class SolutionPrefix:
    def kInversePairs(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        if k > n * (n - 1) // 2:
            return 0
        prev = [1] + [0] * k
        for i in range(2, n + 1):
            pre = [0] + list(accumulate(prev))          # pre[j+1] = prev[0] + ... + prev[j]
            prev = [(pre[j + 1] - pre[max(0, j - i + 1)]) % MOD for j in range(k + 1)]
        return prev[k]

assert SolutionPrefix().kInversePairs(3, 0) == 1
assert SolutionPrefix().kInversePairs(3, 1) == 2
print("SolutionPrefix OK")

SolutionPrefix OK


Ở đây `accumulate` cộng các số đã `< MOD` nên tổng tối đa ~`1000 · 10^9` — Python không tràn, và ta lấy `%` ở bước trừ. Ngắn gọn và nhanh vì `accumulate`, list comprehension chạy ở tốc độ C.

**Độ phức tạp Cách 3:** Time `O(n · k)`, Space `O(k)`.

### 7.3. Cách 4 — Công thức trượt trên một mảng

**Giải thích.** Dùng Insight 3: `cur[j] = cur[j-1] + prev[j] - prev[j-i]`. Thêm hai tối ưu nhỏ:

- Ở hàng `i`, chỉ cần tính `j` tới `min(k, i(i-1)/2)` — các ô sau đều 0.
- Một mảng `prev`, một mảng `cur` (tạo mới mỗi hàng).

In [16]:
def k_inverse_sliding(n: int, k: int) -> int:
    if k > n * (n - 1) // 2:
        return 0
    prev = [1] + [0] * k
    for i in range(2, n + 1):
        cur = [0] * (k + 1)
        top = min(k, i * (i - 1) // 2)
        for j in range(top + 1):
            v = prev[j]
            if j >= 1:
                v += cur[j - 1]
            if j >= i:
                v -= prev[j - i]
            cur[j] = v % MOD
        prev = cur
    return prev[k]

print(k_inverse_sliding(3, 0), k_inverse_sliding(3, 1), k_inverse_sliding(4, 3), k_inverse_sliding(10, 20))
assert k_inverse_sliding(10, 20) == k_inverse_naive(10, 20)

1 2 6 230131


**Đọc code:**

- `v = prev[j]` (ô mới vào cửa sổ) `+ cur[j-1]` (cửa sổ trước) `- prev[j-i]` (ô rời cửa sổ).
- `top`: số nghịch thế tối đa của hoán vị `i` phần tử; ô lớn hơn giữ 0.
- `cur[j] = v % MOD` đưa giá trị (có thể âm) về đúng khoảng.

**Code nộp LeetCode (Cách 4):**

In [17]:
class SolutionSliding:
    def kInversePairs(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        if k > n * (n - 1) // 2:
            return 0
        dp = [1] + [0] * k
        for i in range(2, n + 1):
            new = [0] * (k + 1)
            run = 0
            for j in range(min(k, i * (i - 1) // 2) + 1):
                run += dp[j]                    # ô vào cửa sổ
                if j >= i:
                    run -= dp[j - i]            # ô rời cửa sổ
                new[j] = run % MOD
            dp = new
        return dp[k]

assert SolutionSliding().kInversePairs(3, 1) == 2
assert SolutionSliding().kInversePairs(1000, 1000) == SolutionPrefix().kInversePairs(1000, 1000)
print("SolutionSliding OK")

SolutionSliding OK


Bản nộp viết theo kiểu "biến chạy" `run` = tổng cửa sổ hiện tại — tương đương `cur[j-1]` nhưng không cần đọc mảng. `run` có thể lớn dần (không lấy mod) nhưng tối đa cỡ `k · MOD`, vẫn nhỏ với Python.

**Độ phức tạp Cách 4:** Time `O(n · k)`, Space `O(k)`.

## 8. TRACE từng bước

Trace **Cách 4** với `n = 4`, `k = 6`. Ở mỗi hàng, cửa sổ dài `i` trượt trên hàng trước.

**Hàng `i = 3`** từ `prev = [1, 1, 0, 0, 0, 0, 0]` (hàng 2):

| j | + prev[j] | − prev[j−3] | run | cur[j] |
|---|---|---|---|---|
| 0 | +1 | — | 1 | 1 |
| 1 | +1 | — | 2 | 2 |
| 2 | +0 | — | 2 | 2 |
| 3 | +0 | −prev[0] = −1 | 1 | 1 |

(`top = 3`, các ô sau là 0.)

**Hàng `i = 4`** từ `prev = [1, 2, 2, 1, 0, 0, 0]`:

| j | + prev[j] | − prev[j−4] | run | cur[j] |
|---|---|---|---|---|
| 0 | +1 | — | 1 | 1 |
| 1 | +2 | — | 3 | 3 |
| 2 | +2 | — | 5 | 5 |
| 3 | +1 | — | 6 | 6 |
| 4 | +0 | −1 | 5 | 5 |
| 5 | +0 | −2 | 3 | 3 |
| 6 | +0 | −2 | 1 | 1 |

In [18]:
def trace_sliding(n, k):
    dp = [1] + [0] * k
    print(f"i=1: {dp}")
    for i in range(2, n + 1):
        new = [0] * (k + 1)
        run = 0
        print(f"--- hàng i={i} (cửa sổ dài {i}), prev = {dp}")
        for j in range(min(k, i * (i - 1) // 2) + 1):
            run += dp[j]
            out = ""
            if j >= i:
                run -= dp[j - i]
                out = f", bớt prev[{j - i}]={dp[j - i]}"
            new[j] = run
            print(f"   j={j}: thêm prev[{j}]={dp[j]}{out} -> run={run}")
        dp = new
        print(f"i={i}: {dp}")
    return dp[k]

assert trace_sliding(4, 6) == 1
assert [k_inverse_sliding(4, j) for j in range(7)] == [1, 3, 5, 6, 5, 3, 1]

i=1: [1, 0, 0, 0, 0, 0, 0]
--- hàng i=2 (cửa sổ dài 2), prev = [1, 0, 0, 0, 0, 0, 0]
   j=0: thêm prev[0]=1 -> run=1
   j=1: thêm prev[1]=0 -> run=1
i=2: [1, 1, 0, 0, 0, 0, 0]
--- hàng i=3 (cửa sổ dài 3), prev = [1, 1, 0, 0, 0, 0, 0]
   j=0: thêm prev[0]=1 -> run=1
   j=1: thêm prev[1]=1 -> run=2
   j=2: thêm prev[2]=0 -> run=2
   j=3: thêm prev[3]=0, bớt prev[0]=1 -> run=1
i=3: [1, 2, 2, 1, 0, 0, 0]
--- hàng i=4 (cửa sổ dài 4), prev = [1, 2, 2, 1, 0, 0, 0]
   j=0: thêm prev[0]=1 -> run=1
   j=1: thêm prev[1]=2 -> run=3
   j=2: thêm prev[2]=2 -> run=5
   j=3: thêm prev[3]=1 -> run=6
   j=4: thêm prev[4]=0, bớt prev[0]=1 -> run=5
   j=5: thêm prev[5]=0, bớt prev[1]=2 -> run=3
   j=6: thêm prev[6]=0, bớt prev[2]=2 -> run=1
i=4: [1, 3, 5, 6, 5, 3, 1]


Kiểm tra lại bằng liệt kê thật: các hoán vị của `1..4` có đúng 5 nghịch thế:

In [19]:
five = [p for p in permutations(range(1, 5)) if count_inversions(p) == 5]
print(five)
assert len(five) == 3

[(3, 4, 2, 1), (4, 2, 3, 1), (4, 3, 1, 2)]


Trace **Cách 3** (prefix sum) với `n = 4`, `k = 4`: in mảng `pre` và cửa sổ dùng cho từng ô. Đây là cách nhìn "tổng đoạn = hiệu hai prefix".

| hàng i | dp hàng trước | pre | ô j=3 = pre[4] − pre[max(0, 3−i+1)] |
|---|---|---|---|
| 3 | `[1,1,0,0,0]` | `[0,1,2,2,2,2]` | `pre[4] − pre[1] = 2 − 1 = 1` |
| 4 | `[1,2,2,1,0]` | `[0,1,3,5,6,6]` | `pre[4] − pre[0] = 6 − 0 = 6` |

In [20]:
def trace_prefix(n, k):
    dp = [1] + [0] * k
    for i in range(2, n + 1):
        pre = [0] + list(accumulate(dp))
        print(f"hàng i={i}: dp trước = {dp}, pre = {pre}")
        new = []
        for j in range(k + 1):
            lo = max(0, j - i + 1)
            v = pre[j + 1] - pre[lo]
            print(f"    j={j}: cửa sổ dp[{lo}..{j}] = {dp[lo:j + 1]} -> pre[{j + 1}] - pre[{lo}] = {v}")
            new.append(v)
        dp = new
    return dp[k]

assert trace_prefix(4, 4) == 5

hàng i=2: dp trước = [1, 0, 0, 0, 0], pre = [0, 1, 1, 1, 1, 1]
    j=0: cửa sổ dp[0..0] = [1] -> pre[1] - pre[0] = 1
    j=1: cửa sổ dp[0..1] = [1, 0] -> pre[2] - pre[0] = 1
    j=2: cửa sổ dp[1..2] = [0, 0] -> pre[3] - pre[1] = 0
    j=3: cửa sổ dp[2..3] = [0, 0] -> pre[4] - pre[2] = 0
    j=4: cửa sổ dp[3..4] = [0, 0] -> pre[5] - pre[3] = 0
hàng i=3: dp trước = [1, 1, 0, 0, 0], pre = [0, 1, 2, 2, 2, 2]
    j=0: cửa sổ dp[0..0] = [1] -> pre[1] - pre[0] = 1
    j=1: cửa sổ dp[0..1] = [1, 1] -> pre[2] - pre[0] = 2
    j=2: cửa sổ dp[0..2] = [1, 1, 0] -> pre[3] - pre[0] = 2
    j=3: cửa sổ dp[1..3] = [1, 0, 0] -> pre[4] - pre[1] = 1
    j=4: cửa sổ dp[2..4] = [0, 0, 0] -> pre[5] - pre[2] = 0
hàng i=4: dp trước = [1, 2, 2, 1, 0], pre = [0, 1, 3, 5, 6, 6]
    j=0: cửa sổ dp[0..0] = [1] -> pre[1] - pre[0] = 1
    j=1: cửa sổ dp[0..1] = [1, 2] -> pre[2] - pre[0] = 3
    j=2: cửa sổ dp[0..2] = [1, 2, 2] -> pre[3] - pre[0] = 5
    j=3: cửa sổ dp[0..3] = [1, 2, 2, 1] -> pre[4] - pre[0] = 6
    

## 9. Code nộp LeetCode

Chọn Cách 3 (prefix sum với `accumulate`): ngắn, nhanh trong Python nhờ các hàm dựng sẵn, dễ chứng minh.

In [21]:
from itertools import accumulate

class Solution:
    def kInversePairs(self, n: int, k: int) -> int:
        MOD = 10**9 + 7
        if k > n * (n - 1) // 2:                 # vượt số nghịch thế tối đa
            return 0
        dp = [1] + [0] * k                       # i = 1: chỉ hoán vị [1], 0 nghịch thế
        for i in range(2, n + 1):
            pre = [0] + list(accumulate(dp))     # pre[j+1] = dp[0] + ... + dp[j]
            # dp mới[j] = tổng dp cũ trên cửa sổ [j-i+1, j] (cắt ở 0)
            dp = [(pre[j + 1] - pre[max(0, j - i + 1)]) % MOD for j in range(k + 1)]
        return dp[k]

sol = Solution()
assert sol.kInversePairs(3, 0) == 1
assert sol.kInversePairs(3, 1) == 2
t = time.perf_counter()
print("n=k=1000:", sol.kInversePairs(1000, 1000), f"({time.perf_counter() - t:.2f}s)")

n=k=1000: 663677020 (0.11s)


## 10. Vì sao thuật toán đúng

**Bổ đề (song ánh chèn).** Ánh xạ: (hoán vị `q` của `1..i-1`, số `t ∈ [0, i-1]`) ↦ hoán vị `p` của `1..i` nhận được bằng cách chèn `i` vào vị trí cách cuối `t` ô. Đây là **song ánh**: từ `p`, xóa số `i` ta được `q` và `t` = số phần tử đứng sau `i`.

**Đếm nghịch thế.** `inv(p) = inv(q) + t`: các cặp không chứa `i` giữ nguyên thứ tự tương đối như trong `q`; các cặp chứa `i` là nghịch thế ⇔ phần tử kia đứng **sau** `i` (vì `i` lớn nhất) — có đúng `t` phần tử như vậy.

**Suy ra công thức.** Số hoán vị `p` của `1..i` có `inv(p) = j` bằng số cặp `(q, t)` với `inv(q) = j - t`, `0 <= t <= i-1`, `t <= j`:

```text
dp[i][j] = Σ_{t=0}^{min(j, i-1)} dp[i-1][j-t]
```

Quy nạp theo `i` với cơ sở `dp[1] = [1, 0, 0, ...]` → mọi giá trị đúng. Cách 3 và 4 chỉ là cách tính **cùng tổng đó** nhanh hơn (prefix sum, cửa sổ trượt), và mọi phép toán thực hiện modulo `10^9 + 7` đều bảo toàn đồng dư.

## 11. Phân tích Time/Space

| Cách | Time | Space | `n = k = 1000` |
|---|---|---|---|
| 1. Brute force | `O(n! · n^2)` | `O(n)` | không thể |
| 2. DP trực tiếp | `O(n · k · min(n, k))` | `O(k)` | ~`5·10^8` phép — TLE |
| 3. Prefix sum | `O(n · k)` | `O(k)` | `10^6` — nhanh |
| 4. Cửa sổ trượt | `O(n · k)` | `O(k)` | `10^6` — nhanh |

In [22]:
for name, cls in [("Prefix", SolutionPrefix), ("Sliding", SolutionSliding), ("Solution", Solution)]:
    t = time.perf_counter()
    r = cls().kInversePairs(1000, 1000)
    print(f"{name:>9}: {r}  {time.perf_counter() - t:.3f}s")
t = time.perf_counter()
SolutionNaive().kInversePairs(100, 100)
print(f"    Naive: n=k=100 mất {time.perf_counter() - t:.3f}s (n=k=1000 sẽ chậm hơn ~1000 lần)")

   Prefix: 663677020  0.117s
  Sliding: 663677020  0.060s


 Solution: 663677020  0.113s
    Naive: n=k=100 mất 0.011s (n=k=1000 sẽ chậm hơn ~1000 lần)


## 12. Bộ test

In [23]:
impls = {
    "naive": k_inverse_naive,
    "prefix": k_inverse_prefix,
    "sliding": k_inverse_sliding,
    "SolutionNaive": lambda n, k: SolutionNaive().kInversePairs(n, k),
    "SolutionPrefix": lambda n, k: SolutionPrefix().kInversePairs(n, k),
    "SolutionSliding": lambda n, k: SolutionSliding().kInversePairs(n, k),
    "Solution": lambda n, k: Solution().kInversePairs(n, k),
}

# Ví dụ đề
for name, f in impls.items():
    assert f(3, 0) == 1 and f(3, 1) == 2, name

# Edge case
edge = [(1, 0, 1), (1, 1, 0), (2, 1, 1), (2, 2, 0), (4, 6, 1), (4, 7, 0), (5, 10, 1), (1000, 0, 1)]
for n, k, exp in edge:
    for name, f in impls.items():
        if name in ("naive", "SolutionNaive") and n > 50:
            continue
        assert f(n, k) == exp, (name, n, k)
print("Ví dụ đề + edge case OK")

# Đối chiếu brute force: mọi n <= 7, mọi k <= 22
from collections import Counter
for n in range(1, 8):
    cnt = Counter(count_inversions(p) for p in permutations(range(1, n + 1)))
    for k in range(23):
        for name, f in impls.items():
            assert f(n, k) == cnt[k], (name, n, k)
print("Khớp brute force với mọi n <= 7, k <= 22")

# Ngẫu nhiên có seed: các cách DP khớp nhau
import random
random.seed(629)
for _ in range(40):
    n, k = random.randint(1, 60), random.randint(0, 200)
    vals = {f(n, k) for f in impls.values()}
    assert len(vals) == 1, (n, k, vals)
for _ in range(5):
    n, k = random.randint(500, 1000), random.randint(0, 1000)
    assert k_inverse_prefix(n, k) == k_inverse_sliding(n, k) == Solution().kInversePairs(n, k)
print("Test ngẫu nhiên khớp")

# Tính chất: tổng một hàng = n!, và hàng đối xứng
import math
for n in range(1, 9):
    M = n * (n - 1) // 2
    row = [k_inverse_sliding(n, j) for j in range(M + 1)]
    assert sum(row) == math.factorial(n) and row == row[::-1]
print("Tổng hàng = n! và tính đối xứng: OK")

Ví dụ đề + edge case OK
Khớp brute force với mọi n <= 7, k <= 22


Test ngẫu nhiên khớp
Tổng hàng = n! và tính đối xứng: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cửa sổ dài sai (`i - 1` thay vì `i` vị trí chèn)

In [24]:
def wrong_window(n, k):
    dp = [1] + [0] * k
    for i in range(2, n + 1):
        pre = [0] + list(accumulate(dp))
        dp = [(pre[j + 1] - pre[max(0, j - i + 2)]) % MOD for j in range(k + 1)]   # SAI: cửa sổ dài i-1
    return dp[k]

print("n=3,k=1 sai:", wrong_window(3, 1), "| đúng:", Solution().kInversePairs(3, 1))

n=3,k=1 sai: 1 | đúng: 2


**Vì sao sai:** số `i` có **`i`** vị trí chèn (`t = 0..i-1`), nên cửa sổ dài `i`: `[j - i + 1, j]`. Bớt một vị trí là mất hoán vị (ví dụ mất trường hợp chèn lên đầu).

**Code đúng:** `pre[max(0, j - i + 1)]`.

### Lỗi 2 — Quên lấy modulo sau phép trừ trong ngôn ngữ khác / lấy sai thứ tự

In [25]:
def wrong_mod(n, k):
    dp = [1] + [0] * k
    for i in range(2, n + 1):
        pre = [0]
        for x in dp:
            pre.append((pre[-1] + x) % MOD)
        # SAI (mô phỏng C++/Java): lấy pre[j+1] % MOD - pre[lo] % MOD rồi không đưa về [0, MOD)
        dp = [pre[j + 1] % MOD - pre[max(0, j - i + 1)] % MOD for j in range(k + 1)]
    return dp[k]

bad = [(k, wrong_mod(30, k), Solution().kInversePairs(30, k)) for k in range(150) if wrong_mod(30, k) != Solution().kInversePairs(30, k)]
print(f"n=30: {len(bad)} giá trị k cho kết quả sai, ví dụ (k, sai, đúng):", bad[:3])
assert bad and all(v < 0 for _, v, _ in bad)

n=30: 55 giá trị k cho kết quả sai, ví dụ (k, sai, đúng): [(41, -368732605, 631267402), (42, -627796903, 372203104), (45, -614087139, 385912868)]


**Vì sao sai:** prefix sum đã lấy modulo nên `pre[j+1]` có thể **nhỏ hơn** `pre[lo]` dù tổng thật lớn hơn → hiệu âm. Trong Python, nếu không bọc `% MOD` ở ngoài, giá trị âm lan dần sang các hàng sau.

**Code đúng:** `(pre[j+1] - pre[lo]) % MOD` (C++/Java: `((a - b) % MOD + MOD) % MOD`).

### Lỗi 3 — Cập nhật tại chỗ trên cùng một mảng theo chiều tăng `j`

In [26]:
def wrong_inplace(n, k):
    dp = [1] + [0] * k
    for i in range(2, n + 1):
        for j in range(k + 1):
            dp[j] = sum(dp[j - t] for t in range(min(j, i - 1) + 1)) % MOD   # SAI: dp[j-t] đã là hàng mới
    return dp[k]

print("n=3,k=2 sai:", wrong_inplace(3, 2), "| đúng:", Solution().kInversePairs(3, 2))

n=3,k=2 sai: 4 | đúng: 2


**Vì sao sai:** khi tính `dp[j]` của hàng `i`, các ô `dp[j-1], dp[j-2], ...` **đã bị ghi đè** bằng giá trị hàng `i`, trong khi công thức cần hàng `i-1`.

**Code đúng:** dùng mảng mới cho mỗi hàng (hoặc duyệt `j` giảm dần nếu công thức cho phép — ở đây không đơn giản vì cửa sổ nằm bên trái).

### Lỗi 4 — Không trả sớm khi `k` quá lớn, rồi truy cập ngoài mảng ở cách khác

Nếu cài đặt chỉ cấp phát mảng tới `n(n-1)/2` (để tiết kiệm) mà quên kiểm tra `k > n(n-1)/2`, `dp[k]` sẽ `IndexError`. Cách an toàn: `if k > n*(n-1)//2: return 0` ở đầu hàm — đúng về toán và tránh lỗi chỉ số.

## 14. Mẹo & nhận dạng pattern

- **"Đếm số hoán vị/dãy có đúng `k` (cặp, đoạn, ...)"** với `n, k ~ 1000` → DP `dp[i][j]`, thường xây bằng cách **chèn phần tử lớn nhất/nhỏ nhất**.
- **Công thức là tổng một đoạn liên tiếp của hàng trước** → **prefix sum** hoặc **cửa sổ trượt**: giảm một chiều độ phức tạp.
- `n · k ≤ 10^6` trong Python: nên dùng `accumulate` + list comprehension để chạy nhanh.
- **Modulo + phép trừ**: luôn `% MOD` ở ngoài hiệu.
- Kiểm tra tính chất toàn cục (tổng hàng = `n!`, đối xứng) là cách bắt lỗi rất rẻ.

**Cùng một mẹo ở bài khác — Number of Dice Rolls With Target Sum (LeetCode 1155).** Tung `d` con xúc xắc `f` mặt, đếm số cách tổng bằng `target`. Công thức `ways[i][s] = Σ_{x=1}^{f} ways[i-1][s-x]` cũng là **tổng một cửa sổ dài `f`** của hàng trước → prefix sum cho `O(d · target)`. Nhận ra "tổng cửa sổ" là nhận ra bài.

In [27]:
def dice_ways(d, f, target):
    ways = [1] + [0] * target
    for _ in range(d):
        pre = [0] + list(accumulate(ways))
        ways = [(pre[s] - pre[max(0, s - f)]) % MOD for s in range(target + 1)]   # cửa sổ ways[s-f .. s-1]
    return ways[target]

from itertools import product as _product
for d, f, target in [(1, 6, 3), (2, 6, 7), (3, 4, 6), (4, 3, 8)]:
    brute = sum(1 for roll in _product(range(1, f + 1), repeat=d) if sum(roll) == target)
    assert dice_ways(d, f, target) == brute
print("dice_ways(30, 30, 500) =", dice_ways(30, 30, 500))

dice_ways(30, 30, 500) = 222616187


## 15. Biến thể

1. **Hoán vị có nhiều nhất `k` nghịch thế:** tổng `dp[n][0..k]` (Bài tập 5).
2. **Đếm nghịch thế của một mảng cho trước (`n` lớn):** merge sort hoặc Fenwick tree `O(n log n)`.
3. **Hàm sinh:** `Σ_j dp[n][j] x^j = Π_{i=1}^{n} (1 + x + ... + x^{i-1})` — Cách 2 chính là nhân đa thức từng bước.
4. **Count Inversions with constraints (LeetCode 3193):** thêm yêu cầu số nghịch thế của từng tiền tố — cùng DP chèn phần tử.
5. **Hoán vị có đúng `k` "descent" (số Euler):** công thức DP khác nhưng cùng tư duy chèn.

In [28]:
# Biến thể 3: kiểm tra bằng nhân đa thức
def poly_mul(a, b, k):
    res = [0] * (k + 1)
    for i, x in enumerate(a):
        if x == 0 or i > k:
            continue
        for j, y in enumerate(b):
            if i + j > k:
                break
            res[i + j] = (res[i + j] + x * y) % MOD
    return res

def k_inverse_poly(n, k):
    poly = [1] + [0] * k
    for i in range(1, n + 1):
        poly = poly_mul(poly, [1] * i, k)
    return poly[k]

assert all(k_inverse_poly(n, k) == Solution().kInversePairs(n, k) for n in range(1, 12) for k in range(30))
print("Hàm sinh khớp với DP; n=10, k=20:", k_inverse_poly(10, 20))

Hàm sinh khớp với DP; n=10, k=20: 230131


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | liệt kê `n!` hoán vị | `O(n! · n^2)` | `O(n)` | kiểm chứng `n <= 8` |
| 2. DP trực tiếp | chèn số `i`, cộng `i` ô hàng trước | `O(n · k · min(n,k))` | `O(k)` | hiểu công thức |
| 3. Prefix sum | tổng đoạn = hiệu hai prefix | `O(n · k)` | `O(k)` | **nộp bài** (nhanh với `accumulate`) |
| 4. Cửa sổ trượt | `run += vào − ra` | `O(n · k)` | `O(k)` | khi muốn ít bộ nhớ tạm, ngôn ngữ khác |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `kInversePairs(4, 2)`, `kInversePairs(5, 0)`, `kInversePairs(3, 4)`?

**Bài 2 (tính tay).** Từ hàng `i = 4` (`[1, 3, 5, 6, 5, 3, 1]`), tính tay `dp[5][0..4]` bằng công thức cửa sổ dài 5.

**Bài 3 (điền TODO).** Viết `count_inversions_merge(arr)` đếm nghịch thế trong `O(n log n)` bằng merge sort.

**Bài 4 (tự viết lại).** Viết lại Cách 4 **top-down** (đệ quy có nhớ) theo công thức trượt: `f(i, j) = f(i, j-1) + f(i-1, j) - f(i-1, j-i)`.

**Bài 5 (mini challenge).** Viết `at_most_k(n, k)` = số hoán vị của `1..n` có **nhiều nhất** `k` nghịch thế, modulo `10^9+7`, trong `O(n · k)`.

In [29]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def count_inversions_merge(arr):
    # TODO: hàm đệ quy trả về (mảng đã sắp, số nghịch thế); khi trộn, nếu lấy phần tử bên phải
    #       thì cộng số phần tử còn lại bên trái
    pass

# Bài 4 — TODO
def k_inverse_topdown(n, k):
    # TODO: lru_cache; f(i, j) = 0 nếu j < 0; f(1, j) = 1 nếu j == 0 ngược lại 0
    pass

# Bài 5 — TODO
def at_most_k(n, k):
    # TODO: tính hàng dp[n] như Solution rồi cộng dp[0..k]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: dùng hàng `i = 4` ở mục 4.3.
- Mức 2: `k = 0` luôn có đúng một hoán vị.
- Mức 3: `n = 3` có tối đa 3 nghịch thế.

**Bài 2.**
- Mức 1: `dp[5][j] = Σ dp[4][j-4..j]`.
- Mức 2: `dp[5][0] = 1`, `dp[5][1] = 1 + 3`.
- Mức 3: `dp[5][4]` = tổng `dp[4][0..4]` = `1 + 3 + 5 + 6 + 5` (cửa sổ dài 5 kết thúc ở cột 4).

**Bài 3.**
- Mức 1: chia đôi, đếm hai nửa đệ quy.
- Mức 2: khi trộn, `left[i] > right[j]` thì mọi phần tử `left[i:]` đều lớn hơn `right[j]`.
- Mức 3: cộng `len(left) - i`.

**Bài 4.**
- Mức 1: cơ sở `f(1, 0) = 1`.
- Mức 2: `j < 0` trả 0; `j > i(i-1)/2` trả 0 (tùy chọn).
- Mức 3: nâng `sys.setrecursionlimit` vì độ sâu tới `n + k`.

**Bài 5.**
- Mức 1: tổng tiền tố của hàng cuối.
- Mức 2: `pre[k+1]` của hàng `n`.
- Mức 3: đối chiếu brute force với `n <= 7`.

## 19. Đáp án tham khảo

**Bài 1:** `5`, `1`, `0`.

**Bài 2:** `dp[5][0..4] = [1, 4, 9, 15, 20]` (`dp[5][4] = 1 + 3 + 5 + 6 + 5 = 20`).

In [30]:
assert Solution().kInversePairs(4, 2) == 5
assert Solution().kInversePairs(5, 0) == 1
assert Solution().kInversePairs(3, 4) == 0
assert [Solution().kInversePairs(5, j) for j in range(5)] == [1, 4, 9, 15, 20]

# Bài 3
def count_inversions_merge(arr):
    def solve(a):
        if len(a) <= 1:
            return a, 0
        mid = len(a) // 2
        L, x = solve(a[:mid])
        R, y = solve(a[mid:])
        merged, i, j, inv = [], 0, 0, x + y
        while i < len(L) and j < len(R):
            if L[i] <= R[j]:
                merged.append(L[i]); i += 1
            else:
                merged.append(R[j]); j += 1
                inv += len(L) - i
        merged += L[i:] + R[j:]
        return merged, inv
    return solve(list(arr))[1]

random.seed(3)
for _ in range(300):
    arr = [random.randint(0, 20) for _ in range(random.randint(0, 30))]
    assert count_inversions_merge(arr) == count_inversions(arr)
print("Bài 1, 2, 3 OK")

Bài 1, 2, 3 OK


In [31]:
# Bài 4
import sys
from functools import lru_cache
sys.setrecursionlimit(10000)

def k_inverse_topdown(n, k):
    @lru_cache(maxsize=None)
    def f(i, j):
        if j < 0 or j > i * (i - 1) // 2:
            return 0
        if i == 1:
            return 1 if j == 0 else 0
        if j == 0:
            return 1
        return (f(i, j - 1) + f(i - 1, j) - f(i - 1, j - i)) % MOD
    return f(n, k)

assert all(k_inverse_topdown(n, k) == Solution().kInversePairs(n, k) for n in range(1, 15) for k in range(40))
assert k_inverse_topdown(200, 300) == Solution().kInversePairs(200, 300)

# Bài 5
def at_most_k(n, k):
    dp = [1] + [0] * k
    for i in range(2, n + 1):
        pre = [0] + list(accumulate(dp))
        dp = [(pre[j + 1] - pre[max(0, j - i + 1)]) % MOD for j in range(k + 1)]
    return sum(dp) % MOD

for n in range(1, 8):
    invs = [count_inversions(p) for p in permutations(range(1, n + 1))]
    for k in range(15):
        assert at_most_k(n, k) == sum(1 for v in invs if v <= k)
print("Bài 4, 5 OK")

Bài 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "DP tổng cửa sổ bằng prefix sum":**

```python
dp = [1] + [0] * k                         # hàng cơ sở
for i in range(2, n + 1):
    pre = [0] + list(accumulate(dp))       # pre[j+1] = dp[0..j]
    dp = [(pre[j + 1] - pre[max(0, j - W(i) + 1)]) % MOD for j in range(k + 1)]
return dp[k]
```

Với bài này độ dài cửa sổ `W(i) = i` (số vị trí chèn số `i`).

**Câu thần chú:** *"Chèn số lớn nhất: thêm 0..i-1 nghịch thế; tổng cửa sổ dài i; prefix sum cho O(1)."*

**Checklist trước khi submit:**

- [ ] Cửa sổ dài **`i`**: `[j - i + 1, j]`, cắt ở 0.
- [ ] Hàng cơ sở `i = 1`: `[1, 0, 0, ...]`.
- [ ] `% MOD` **sau** phép trừ.
- [ ] Trả sớm 0 khi `k > n(n-1)/2`.
- [ ] Mỗi hàng dùng mảng mới (không ghi đè hàng trước khi còn cần).
- [ ] Thử `(3,0) → 1`, `(3,1) → 2`, `(4,3) → 6`, `(1,1) → 0`.
- [ ] Đối chiếu brute force với `n <= 7`.